# Aptus-R · Phase 0 EDA (T0.3)

**Status: data-ready — not yet executed.** The 100K dataset (`data/candidates.jsonl.gz`) is not in this repo; drop the organizer file there and *Run All* to populate every analysis below. No numbers are hard-coded — every figure is computed live from the data.

Goal (per [PHASE_0 §T0.3](../phases/PHASE_0_foundations.md)): confirm the pool facts that justify every weight in `jd_requirements.yaml`:

1. Title distribution → tier pyramid
2. Skill frequency → Tier-B/C/D buckets
3. Company landscape (service firms vs product firms)
4. `redrob_signals` distributions (+ `-1` sentinel rates)
5. Honeypot scan — count per rule + examples per rule


In [ ]:
# --- Setup ------------------------------------------------------------------
import sys
from collections import Counter
from datetime import date
from pathlib import Path

import pandas as pd
import yaml

# Repo-root imports (scripts/ + src/) without requiring an installed package.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from aptus import config, honeypot  # noqa: E402
from scripts.dataio import iter_candidates  # noqa: E402

DATA_PATH = ROOT / "data" / "candidates.jsonl"
print("data path:", DATA_PATH, "exists:", DATA_PATH.exists())

# Load all candidates once into a list for repeated analysis. ~100K parsed
# objects fit comfortably under the 16 GB cap; EDA is untimed (Phase A).
candidates = list(iter_candidates(DATA_PATH))
print(f"loaded {len(candidates):,} candidates")

## 1 · Title distribution → tier pyramid

Map each `current_title` to a tier from `config/title_taxonomy.yaml`. Expect a pyramid: many Tier-4/5 (general SWE / non-technical), few Tier-1 (rare senior AI/ML) — the true positives we must rank right.


In [ ]:
taxonomy = yaml.safe_load((ROOT / "config" / "title_taxonomy.yaml").read_text(encoding="utf-8"))

def title_tier(title: str) -> str:
    t = (title or "").lower()
    for tier_name, block in taxonomy["tiers"].items():
        for ex in block["examples"]:
            if ex in t:
                return tier_name
    return "unmapped"

tier_counts = Counter(title_tier(c.profile.current_title) for c in candidates)
tier_order = ["tier_1", "tier_2", "tier_3", "tier_4", "tier_5", "unmapped"]
tier_df = pd.DataFrame(
    [(t, taxonomy["tiers"].get(t, {}).get("label", "-"), tier_counts.get(t, 0)) for t in tier_order],
    columns=["tier", "label", "count"],
)
tier_df["pct"] = (100 * tier_df["count"] / max(len(candidates), 1)).round(1)
display(tier_df)

# Top raw titles for a sanity eyeball
print("Top 25 raw titles:")
display(pd.Series(Counter(c.profile.current_title for c in candidates)).sort_values(ascending=False).head(25))

## 2 · Skill frequency → Tier-B/C/D buckets

Buzzwords appear thousands of times (Tier-B), deep skills ~1k (Tier-C), plain-language / rare skills a handful of times (Tier-D). This justifies S1 semantic retrieval + the concept thesaurus.


In [ ]:
skill_freq = Counter()
for c in candidates:
    for s in c.skills:
        if s.name:
            skill_freq[s.name.strip().lower()] += 1

skill_series = pd.Series(skill_freq).sort_values(ascending=False)
print(f"unique skills: {len(skill_series):,}")
print("\nTop 30 skills (Tier-B buzzword candidates):")
display(skill_series.head(30))

def bucket(freq: int) -> str:
    if freq >= 3000:
        return "Tier-B (buzzword, >=3000)"
    if freq >= 800:
        return "Tier-C (deep, 800-2999)"
    return "Tier-D (plain/rare, <800)"

bucket_counts = Counter(bucket(v) for v in skill_series.values)
display(pd.Series(bucket_counts).rename("n_unique_skills"))

## 3 · Company landscape

Confirm service-firm ubiquity (TCS/Infosys/Wipro/…) vs rare product firms. Drives the `consulting_only` / `no_product_exp` penalties.


In [ ]:
service_firms = {f.lower() for f in config.PENALTIES["service_firms"]}

def is_service(name: str) -> bool:
    n = (name or "").lower()
    return any(sf in n for sf in service_firms)

company_freq = Counter()
for c in candidates:
    if c.profile.current_company:
        company_freq[c.profile.current_company.strip()] += 1

company_series = pd.Series(company_freq).sort_values(ascending=False)
print(f"unique current companies: {len(company_series):,}")
print("\nTop 25 current companies:")
display(company_series.head(25))

n_service_current = sum(1 for c in candidates if is_service(c.profile.current_company))
print(f"\ncandidates currently at a known service firm: {n_service_current:,} "
      f"({100*n_service_current/max(len(candidates),1):.1f}%)")

# Candidates whose ENTIRE career is service-firm (consulting_only penalty trigger)
def all_service(c):
    hist = [e.company for e in c.career_history if e.company]
    return bool(hist) and all(is_service(co) for co in hist)

n_all_service = sum(1 for c in candidates if all_service(c))
print(f"candidates with 100% service-firm career (consulting_only trigger): {n_all_service:,}")

## 4 · `redrob_signals` distributions + sentinel rates

Median `last_active` days, `recruiter_response_rate`, and the `-1` sentinel rates for `github_activity_score` / `offer_acceptance_rate`. These calibrate S4/S5 and the DR-1 sentinel rules.


In [ ]:
REF = config.REFERENCE_DATE

def days_inactive(c):
    try:
        return (REF - date.fromisoformat(c.redrob_signals.last_active_date)).days
    except Exception:
        return None

rows = []
for c in candidates:
    s = c.redrob_signals
    rows.append({
        "days_inactive": days_inactive(c),
        "recruiter_response_rate": s.recruiter_response_rate,
        "open_to_work": s.open_to_work_flag,
        "notice_period_days": s.notice_period_days,
        "github_is_sentinel": s.github_activity_score == -1,
        "offer_is_sentinel": s.offer_acceptance_rate == -1,
        "completeness": s.profile_completeness_score,
        "verified_email": s.verified_email,
        "verified_phone": s.verified_phone,
    })
sig = pd.DataFrame(rows)
display(sig.describe(include="all").T)

print("median days_inactive:", sig["days_inactive"].median())
print("median recruiter_response_rate:", sig["recruiter_response_rate"].median())
print("open_to_work rate:", round(sig["open_to_work"].mean(), 3))
print("github -1 sentinel rate:", round(sig["github_is_sentinel"].mean(), 3))
print("offer  -1 sentinel rate:", round(sig["offer_is_sentinel"].mean(), 3))

## 5 · Honeypot scan (feeds T0.4)

Run the 6-rule gate over the full pool. Expect a sane flag count (~70–90, **not** thousands). Report per-rule counts and eyeball 5 examples per rule.


In [ ]:
results = [honeypot.check(c) for c in candidates]
flagged = [r for r in results if r.is_honeypot]
print(f"flagged honeypots: {len(flagged):,} / {len(candidates):,} "
      f"({100*len(flagged)/max(len(candidates),1):.2f}%)")

rule_counts = Counter()
for r in flagged:
    for rule in r.rules_fired:
        rule_counts[rule] += 1
display(pd.Series(dict(sorted(rule_counts.items()))).rename("n_candidates_fired"))

# 5 example candidate_ids per rule
for rule in sorted(rule_counts):
    examples = [r.candidate_id for r in flagged if rule in r.rules_fired][:5]
    print(f"{rule}: {examples}")

## Next steps

- Lock the gate flag count into `scripts/honeypot_full_scan.py` (T0.4).
- Carry the title-tier, skill-tier, and signal medians into the WGT stratified sampling (Phase 4).
- Any weight in `jd_requirements.yaml` whose justification is not visible above should be revisited.
